In [ ]:
%gherkin
Feature: Calculate Applied Quantity (apl_qty) for Inventory Transactions

  Background:
    Given the transactions table "purgo_playground.purgo_playground.f_inv_movmnt_apl_qty"
    And the data source is the Unity Catalog schema "purgo_playground"

  Scenario Outline: Calculate apl_qty for Positive Transactions with Condition 1
    Given ref_txn_qty is greater than 0
    And cumulative_txn_qty is greater than or equal to cumulative_ref_ord_sched_qty
    When prior_cumulative_ref_ord_sched_qty is less than prior_cumulative_txn_qty
    Then apl_qty should be set to <ref_ord_sched_qty> minus the difference between <prior_cumulative_txn_qty> and <prior_cumulative_ref_ord_sched_qty>
    Examples:
      | txn_id | ref_txn_qty | cumulative_txn_qty | cumulative_ref_ord_sched_qty | ref_ord_sched_qty |
      | 1      | 50.0        | 100.0              | 90.0                         | 50.0              |
      | 3      | 20.0        | 60.0               | 100.0                        | 30.0              |

  Scenario: Positive Transaction Default Handler for Condition 1
    Given ref_txn_qty is greater than 0
    And cumulative_txn_qty is greater than or equal to cumulative_ref_ord_sched_qty
    When prior_cumulative_ref_ord_sched_qty is not less than prior_cumulative_txn_qty
    Then apl_qty should be set to ref_ord_sched_qty

  Scenario Outline: Calculate apl_qty for Positive Transactions with Condition 2
    Given ref_txn_qty is greater than 0
    And cumulative_ref_ord_sched_qty is greater than or equal to cumulative_txn_qty
    When prior_cumulative_ref_ord_sched_qty is greater than prior_cumulative_txn_qty
    Then apl_qty should be set to <ref_txn_qty> minus the difference between <prior_cumulative_ref_ord_sched_qty> and <prior_cumulative_txn_qty>
    Examples:
      | txn_id | ref_txn_qty | cumulative_txn_qty | cumulative_ref_ord_sched_qty | prior_cumulative_txn_qty | prior_cumulative_ref_ord_sched_qty |
      | 1      | 50.0        | 100.0              | 90.0                         | 40.0                      | 30.0                               |

  Scenario: Positive Transaction Default Handler for Condition 2
    Given ref_txn_qty is greater than 0
    And cumulative_ref_ord_sched_qty is greater than or equal to cumulative_txn_qty
    When prior_cumulative_ref_ord_sched_qty is not greater than prior_cumulative_txn_qty
    Then apl_qty should be set to ref_txn_qty

  Scenario Outline: Calculate apl_qty for Negative Transactions with Condition 3
    Given ref_txn_qty is less than 0
    And cumulative_txn_qty is not equal to 0
    And cumulative_ref_ord_sched_qty is greater than 0
    Then apl_qty should be set to ref_txn_qty
    Examples:
      | txn_id | ref_txn_qty | cumulative_txn_qty | cumulative_ref_ord_sched_qty |
      | 2      | -10.0       | 80.0               | 70.0                         |

  Scenario: Default Case Handling
    Given none of the conditions are met
    Then apl_qty should be set to NULL

  Scenario: Error Handling for Invalid Data
    Given ref_txn_qty is NULL
    When calculating apl_qty
    Then an error message "ref_txn_qty cannot be NULL" should be displayed
